 <a target="_blank" href="https://colab.research.google.com/github/mbari-org/stm/blob/main/stm/notebooks/train_topic_model_perch2.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
 Author: Danelle Cline dcline@mbari.org

## Train a topic model with Perch2 embeddings to uncover sound phrases
---
The Perch2 model is a powerful model for capturing rich features in audio sequences.  This notebook demonstrates the application of the Perch2 model on a short sequence of a humpback sond from *Pacific Ocean Sound* [2] data.  Here, we see it captures the song phrases quite well.

### Load configuration
Create the output directory, then load and verify `config.yaml`. The config supplies Perch2 window settings, topic-model hyperparameters, and paths used in later cells.

In [ ]:
from pathlib import Path

output_path = Path("output_perch2")
dataset_path = Path("dataset")
output_path.mkdir(parents=True, exist_ok=True)

from stm.config import Config
try:
    config = Config.load(
        Path("../config.yaml"),
        wav_path=dataset_path,
        output_path=output_path,
    )
    config.verify()
except Exception as e:
    print(e)

### Extract Perch2 embeddings
Measure the dataset duration, build a regular time grid from the configured hop and window, and embed each window with the Perch2 ONNX model.

In [ ]:
from stm.embed import total_audio_seconds
from stm.features import Perch2Extractor, TimeGrid

duration = total_audio_seconds(dataset_path)
n_windows = int((duration - config.perch_audio_seconds) // config.perch_hop_seconds) + 1
print(f"Processing {duration} total audio seconds")

perch_grid = TimeGrid.regular(
    start=0.0,
    hop=config.perch_hop_seconds,
    window=config.perch_audio_seconds,
    n=n_windows,
)
perch_block = Perch2Extractor.from_config(config, model=Path("./perch_v2.onnx")).extract(dataset_path, perch_grid)

### Train the topic model
Fit a topic model on the Perch2 feature block. Document duration matches the Perch2 audio window so each document covers one embedding.

In [ ]:
from stm.topicmodel import TopicModelRunner

runner = TopicModelRunner(timeout=600,
                          gamma=config.gamma,
                          alpha=config.alpha,
                          beta=config.beta,
                          num_topics=13,
                          document_seconds=config.perch_audio_seconds)
result = runner.run_from_block(
    [perch_block],
    config.doc_path,
    config.model_path,
    target=perch_block.grid,
)

### Report model diagnostics
Print average perplexity and the path to the maximum-likelihood topic-over-time file.

In [ ]:
print(result.avg_perplexity, result.maxlikelihood_with_time_path)

### Plot topics on spectrograms
Overlay inferred topics on spectrogram chunks of the source audio for visual inspection.

In [ ]:
from stm.topicmodel.plotter import Plotter

plotter = Plotter(config.model_path, config=config)
plotter.plot(dataset_path, n_chunks=10, chunk_size=60, freq_range=(0, 4000), window_size=1024)